# MMseqs2 CTS Demo

Runs MMseqs2 `easy-cluster` on protein FASTAs from 4 test genomes via the CDM Task Service, clustering all proteins together across genomes.

- **Image:** `ghcr.io/kbaseincubator/cdm_mmseqs2:0.1.0`
- **Mode:** `easy-cluster`. All-vs-all clustering, no reference DB needed
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/mmseqs2/demo_20261106`

Re-executed 2026-11-06 after the tool audit: inputs are now the protein `.faa` files (the earlier run clustered nucleotide genomes and a FASTQ), the tmp dir is outside `/out`, and the checks verify every input protein lands in exactly one cluster.

See the [cdm_mmseqs2 repo](https://github.com/kbaseincubator/cdm_mmseqs2) for details.

## 1. Setup clients

In [1]:
tscli = get_task_service_client()
mincli = get_minio_client()
print(tscli.whoami())

{'user': 'jplfaria', 'roles': ['cts_user'], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input files

Protein FASTAs (`.faa`) for 4 test genomes, already in object storage with CRC64NVME checksums.

In [2]:
input_files = sorted(
    f"cts/{o.object_name}"
    for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True)
    if o.object_name.endswith(".faa")
)
n_input = {}
for f in input_files:
    raw = mincli.get_object(*f.split("/", 1)).read().decode("utf-8")
    n_input[f] = sum(1 for line in raw.splitlines() if line.startswith(">"))
print(f"{len(input_files)} input files, {sum(n_input.values()):,} proteins:")
for f, n in n_input.items():
    print(f"  {n:>5}  {f}")


4 input files, 5,802 proteins:
    583  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa
   3001  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000010565.1_ASM1056v1_genomic.faa
   1999  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000145985.1_ASM14598v1_genomic.faa
    219  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000147015.1_ASM14701v1_genomic.faa


## 3. Submit MMseqs2 easy-cluster job

In [3]:
IMAGE = "ghcr.io/kbaseincubator/cdm_mmseqs2:0.1.0@sha256:24afa107c0dac1a6f093cc081ec19a577c4d0b156dfa8ee66bb2d3c41b97c082"
OUTPUT_DIR = "cts/io/jplfaria/output/mmseqs2/demo_20261106"

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "easy-cluster",
        tscli.insert_files(),
        "/out/cluster_results",
        "/tmp/mmseqs",                 # scratch outside /out, so it is not uploaded
        "--threads", "4",
    ],
    num_containers=1,              # one container: all genomes clustered together
    cpus=4,
    memory="16GB",
    runtime="PT30M"
)
print("Job ID:", job.id)

Job ID: 9e5ec52c-756d-4768-8c54-11a0182b82b4


## 4. Wait for completion

In [4]:
# Block until the job finishes (CTS keeps container logs only for failed jobs,
# so the checks below validate the output files themselves).
import json
job.wait_for_completion()
status = job.get_job_status()
print("state:", status["state"])
print("exit codes:", job.get_exit_codes().get("exit_codes"))


state: complete


exit codes: [0]


## 5. Inspect output files

In [5]:
outs = job.get_job()["outputs"]
for o in outs:
    print(o["crc64nvme"], o["file"])
print(f"\n{len(outs)} output files; scratch files uploaded: {sum('/tmp/' in o['file'] for o in outs)}")


w6swiukRn9c= cts/io/jplfaria/output/mmseqs2/demo_20261106/0/cluster_results_all_seqs.fasta
srHKU8UKXhA= cts/io/jplfaria/output/mmseqs2/demo_20261106/0/cluster_results_rep_seq.fasta
DW1VXmcZyzU= cts/io/jplfaria/output/mmseqs2/demo_20261106/0/cluster_results_cluster.tsv

3 output files; scratch files uploaded: 0


## 6. View cluster results

`cluster_results_cluster.tsv` columns: `representative`, `member` (one row per input protein).

In [6]:
import io, pandas as pd
cl = next(o["file"] for o in outs if o["file"].endswith("cluster_results_cluster.tsv"))
df = pd.read_csv(io.BytesIO(mincli.get_object(*cl.split("/", 1)).read()), sep="\t", header=None, names=["representative", "member"])
rep_fa = next(o["file"] for o in outs if o["file"].endswith("cluster_results_rep_seq.fasta"))
n_rep_seqs = sum(1 for line in mincli.get_object(*rep_fa.split("/", 1)).read().decode().splitlines() if line.startswith(">"))
sizes = df.groupby("representative").size()
print(f"{len(df):,} memberships, {df['representative'].nunique():,} clusters, {n_rep_seqs:,} representative sequences")
print(f"clusters with more than one member: {(sizes > 1).sum():,} (largest: {sizes.max()})")
df.head(10)


5,802 memberships, 4,558 clusters, 4,558 representative sequences
clusters with more than one member: 729 (largest: 33)


,representative,member
0,AE017199.1_1,AE017199.1_1
1,AE017199.1_2,AE017199.1_2
2,AE017199.1_3,AE017199.1_3
3,AE017199.1_5,AE017199.1_5
4,AE017199.1_6,AE017199.1_6
5,AE017199.1_7,AE017199.1_7
6,AE017199.1_8,AE017199.1_8
7,AE017199.1_9,AE017199.1_9
8,AE017199.1_9,AP009389.1_1334
9,AE017199.1_9,AP009389.1_1379


## 7. End-to-end check

Every input protein must appear exactly once as a member, and the number of clusters must match the representative FASTA.

In [7]:
checks = {
    "job complete, exit 0": job.get_job_status()["state"] == "complete"
                            and job.get_exit_codes().get("exit_codes") == [0],
    "every input protein clustered": len(df) == sum(n_input.values()),
    "each member appears once": df["member"].is_unique,
    "clusters == representative sequences": df["representative"].nunique() == n_rep_seqs,
    "no scratch files uploaded": not any("/tmp/" in o["file"] for o in outs),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off, inspect above.")


  [x] job complete, exit 0
  [x] every input protein clustered
  [x] each member appears once
  [x] clusters == representative sequences
  [x] no scratch files uploaded

All green.
